# Env

In [154]:
%matplotlib inline

import os 
import pandas as pd
import numpy as np
import scipy.stats as sst
import matplotlib.pyplot as plt
import zipfile
import datetime
import re
import pymysql

# MySQL

In [155]:
# sql server info
DB_CONFIG = {
    'host': '31.31.198.65',
    'user': 'u0867732_default',
    'password': 'hFFD!2m6',
    'database': 'u0867732_default',
    'port': 3306,
    'charset': 'utf8'
    }

connection = pymysql.connect(**DB_CONFIG, cursorclass=pymysql.cursors.DictCursor)

In [ ]:
# Create table
sql_table = """
    CREATE TABLE IF NOT EXIST main_data (
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(255) NOT NULL, 
        date DATE)
    """

with connection.cursor() as cursor:
    cursor.execute(sql_table)
    cursor.close()

In [ ]:
# SHOW DATABASES
# SHOW TABLES
# DESCRIBE table
# DROP table

## How to create PROCEDURES (looks unnecessary)?

# fetch{one, all, many}
# execute{'', many}

# connection.commit

In [156]:
with connection.cursor() as cursor:
    sql = 'SHOW TABLES'
    cursor.execute(sql)
    version = cursor.fetchall()
    cursor.close() 
    print(f'{version}')

[{'Tables_in_u0867732_default': 'auth_group'}, {'Tables_in_u0867732_default': 'auth_group_permissions'}, {'Tables_in_u0867732_default': 'auth_permission'}, {'Tables_in_u0867732_default': 'auth_user'}, {'Tables_in_u0867732_default': 'auth_user_groups'}, {'Tables_in_u0867732_default': 'auth_user_user_permissions'}, {'Tables_in_u0867732_default': 'django_admin_log'}, {'Tables_in_u0867732_default': 'django_content_type'}, {'Tables_in_u0867732_default': 'django_migrations'}, {'Tables_in_u0867732_default': 'django_session'}, {'Tables_in_u0867732_default': 'main_data'}, {'Tables_in_u0867732_default': 'main_visitors'}, {'Tables_in_u0867732_default': 'polls_choice'}, {'Tables_in_u0867732_default': 'polls_question'}]


In [126]:
with connection.cursor() as cursor:
    sql = 'INSERT INTO main_data(name, date) VALUES(%s, %s)'
    data = (('Chelik', '1990-12-30'),
            ('Chelik_1', '1991-12-30'),
            ('Chelik_2', '1992-12-30'))
    cursor.executemany(sql, data)
    cursor.close()

In [137]:
with connection.cursor() as cursor:
    sql = 'SELECT * FROM main_data'
    cursor.execute(sql)
    version = cursor.fetchall()
    cursor.close()
    print(f'{version}')

[{'id': 15, 'name': 'Chelik', 'date': datetime.date(1990, 12, 30)}, {'id': 16, 'name': 'Chelik_1', 'date': datetime.date(1991, 12, 30)}, {'id': 17, 'name': 'Chelik_2', 'date': datetime.date(1992, 12, 30)}]


In [133]:
connection.commit()
connection.close()

# Data analysis

In [9]:
# list all files in dir, we need the last one
os.listdir()

['.ipynb_checkpoints',
 'Squirtanalysis.ipynb',
 'WhatsApp Chat - Сквиртология 2.zip']

In [10]:
file_to_extract = os.listdir()[2]

In [11]:
# extract file
with zipfile.ZipFile(file_to_extract, 'r') as f:
    f.extractall()

In [13]:
os.listdir()

['.ipynb_checkpoints',
 'Squirtanalysis.ipynb',
 'WhatsApp Chat - Сквиртология 2.zip',
 '_chat.txt']

In [212]:
# read file
text_file = open('_chat.txt', encoding='utf-8')
df = text_file.read()

#df.replace('\u200e', '')
#df.replace('\u202a+7\xa0915\xa0281‑01‑14\u202c', '')

In [213]:
df

'[02.10.2016, 22:43:09] Сквиртология: \u200eСообщения в этой группе теперь защищены сквозным шифрованием.\n[04.09.2016, 17:49:43] \u200eТим изменил(-а) тему на “Я ебу Али Бабу”\n[02.10.2016, 22:43:09] \u200eТим добавил(-а) \u202a+7\xa0915\xa0281‑01‑14\u202c\n[02.10.2016, 22:43:38] Тим: Детские фотки с цветочками от бабокара стали последней каплей.\n[02.10.2016, 22:43:58] Тим: Открываю нормальную группу с нормальным трешняком\n\u200e[02.10.2016, 22:44:23] Тим: \u200eаудиофайл отсутствует\n\u200e[02.10.2016, 22:44:24] Тим: \u200eаудиофайл отсутствует\n\u200e[02.10.2016, 22:44:25] Тим: \u200eаудиофайл отсутствует\n\u200e[02.10.2016, 22:44:25] Тим: \u200eаудиофайл отсутствует\n[02.10.2016, 22:45:07] Тим: 👆 это то, что сказал бабакар, когда узнал про эту группу\n[02.10.2016, 22:48:21] Токс: А где собственно он?\n[02.10.2016, 23:03:09] Тим: Кто?\n\u200e[02.10.2016, 23:03:50] Тим: \u200eвидео отсутствует\n[02.10.2016, 23:03:55] Тим: Олег, грузи всё сюда!)\n[02.10.2016, 23:11:52] \u200eВы поки

In [208]:
# split string for small messages
df = df.split('\n')

# create a df and change a column name
df = pd.DataFrame(df)
df.columns = ['base_text']

# cleam the text
df.apply(lambda x: x.replace('\u200e', ''))
df.apply(lambda x: x.replace('\u202a+7\xa0915\xa0281‑01‑14\u202c', ''))

# split for time and text
df = df['base_text'].str.split(']', expand=True)

# split time and date
df = pd.concat([df[0].str.split(',', expand=True).loc[:, :1], df[1]], axis=1)

# drop nan
df = df.loc[:11502]

df.loc[:, 0] = df[0].str.replace('[', '')
df.columns = ['date', 'time', 'text']

SyntaxError: invalid syntax (<ipython-input-208-b9e9a7411e96>, line 4)

In [ ]:
# more clean-text
for col in df.columns:
    df[col] = df[col].apply(lambda x: x.replace('\u200e', ''))

In [207]:
df['year'] = df['date'].apply(lambda x: datetime.datetime.strptime(x, '%d.%m.%Y').year)

ValueError: time data '\u200e02.10.2016' does not match format '%d.%m.%Y'

In [191]:
a = df['date'].str.split('.', expand=True)

In [192]:
a[0] = a[0].apply(lambda x: x if x.isdigit() else -1)

In [196]:
index_to_drop = a[a[0] == -1].index

In [198]:
df = df.drop(index_to_drop, axis=0).reset_index(drop=True)

In [206]:
df.head(30)

,date,time,text
0,02.10.2016,22:43:09,Сквиртология: ‎Сообщения в этой группе теперь...
1,04.09.2016,17:49:43,‎Тим изменил(-а) тему на “Я ебу Али Бабу”
2,02.10.2016,22:43:09,‎Тим добавил(-а) ‪+7 915 281‑01‑14‬
3,02.10.2016,22:43:38,Тим: Детские фотки с цветочками от бабокара с...
4,02.10.2016,22:43:58,Тим: Открываю нормальную группу с нормальным ...
5,‎02.10.2016,22:44:23,Тим: ‎аудиофайл отсутствует
6,‎02.10.2016,22:44:24,Тим: ‎аудиофайл отсутствует
7,‎02.10.2016,22:44:25,Тим: ‎аудиофайл отсутствует
8,‎02.10.2016,22:44:25,Тим: ‎аудиофайл отсутствует
9,02.10.2016,22:45:07,"Тим: 👆 это то, что сказал бабакар, когда узна..."
